# EMDB Download Coverage for Training PDBs

This notebook computes:
1. Number of downloaded EMDB maps vs number of unique PDB IDs in the **training** dataset.
2. Average number of downloaded EMDB maps per training PDB that has at least one downloaded EMDB match.

In [1]:
from pathlib import Path
import json
import re

import torch

In [2]:
REPO_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebook" else Path.cwd().resolve()
TRAIN_PT = REPO_ROOT / "voxbind" / "dataset" / "data" / "data_train.pt"
MAPPING_JSON = REPO_ROOT / "notebook" / "data" / "crossdocked_pdb_to_emdb.json"
EMDB_DIR = REPO_ROOT / "notebook" / "data" / "emdb"

if not TRAIN_PT.exists():
    raise FileNotFoundError(f"Training dataset file not found: {TRAIN_PT}")
if not MAPPING_JSON.exists():
    raise FileNotFoundError(f"Mapping file not found: {MAPPING_JSON}")

samples = torch.load(TRAIN_PT, weights_only=False)
with open(MAPPING_JSON) as f:
    crossdocked_to_emdb = json.load(f)

def extract_pdb_id(pocket_id: str) -> str:
    return pocket_id.split("/")[1].split("_")[0].upper()

total_pocket_samples = len(samples)
train_pdb_ids = {extract_pdb_id(pocket_dict["id"]) for pocket_dict, _ in samples}

downloaded_emdb_ids = set()
if EMDB_DIR.exists():
    for p in EMDB_DIR.glob("emd_*.map.gz"):
        m = re.match(r"emd_(\d+)\.map\.gz$", p.name)
        if m:
            downloaded_emdb_ids.add(f"EMD-{m.group(1)}")

print(f"Total training pocket samples: {total_pocket_samples:,}")
print(f"Unique training PDB IDs: {len(train_pdb_ids):,}")
print(f"Downloaded EMDB map files: {len(downloaded_emdb_ids):,}")

Total training pocket samples: 99,981
Unique training PDB IDs: 14,635
Downloaded EMDB map files: 2,382


In [4]:
matched_counts = []
matched_pdb_ids = set()

for pdb_id in sorted(train_pdb_ids):
    mapped_emdb_ids = set(crossdocked_to_emdb.get(pdb_id, []))
    downloaded_for_pdb = mapped_emdb_ids & downloaded_emdb_ids
    if downloaded_for_pdb:
        matched_counts.append(len(downloaded_for_pdb))
        matched_pdb_ids.add(pdb_id)

num_pdb_with_downloaded_match = len(matched_counts)
avg_downloaded_per_matched_pdb = (
    sum(matched_counts) / num_pdb_with_downloaded_match
    if num_pdb_with_downloaded_match > 0
    else 0.0
)

pocket_samples_with_match = sum(
    1
    for pocket_dict, _ in samples
    if extract_pdb_id(pocket_dict["id"]) in matched_pdb_ids
)
pct_pocket_with_match = (
    100.0 * pocket_samples_with_match / total_pocket_samples
    if total_pocket_samples > 0
    else 0.0
)

pct_emdb_vs_pdb = (
    100.0 * len(downloaded_emdb_ids) / len(train_pdb_ids)
    if len(train_pdb_ids) > 0
    else 0.0
)
pct_pdb_with_match = (
    100.0 * num_pdb_with_downloaded_match / len(train_pdb_ids)
    if len(train_pdb_ids) > 0
    else 0.0
)

print("\n=== Requested Statistics ===")
print(
    f"Downloaded EMDB files vs training PDB files: "
    f"{len(downloaded_emdb_ids):,} / {len(train_pdb_ids):,} "
    f"({pct_emdb_vs_pdb:.2f}%)"
)
print(
    f"Training PDBs with >=1 downloaded matching EMDB: "
    f"{num_pdb_with_downloaded_match:,} ({pct_pdb_with_match:.2f}%)"
)
print(f"Average downloaded EMDB per matched training PDB: {avg_downloaded_per_matched_pdb:.3f}")
print(
    f"Training pocket samples whose PDB has >=1 downloaded EMDB: "
    f"{pocket_samples_with_match:,} / {total_pocket_samples:,} ({pct_pocket_with_match:.2f}%)"
)


=== Requested Statistics ===
Downloaded EMDB files vs training PDB files: 2,382 / 14,635 (16.28%)
Training PDBs with >=1 downloaded matching EMDB: 3,348 (22.88%)
Average downloaded EMDB per matched training PDB: 7.647
Training pocket samples whose PDB has >=1 downloaded EMDB: 26,614 / 99,981 (26.62%)
